# Gemma 4 Developer Agent — Phase 0 checks, EXP-000 gold control, baseline eval (4x L4)

One notebook, three switches (`RUN_PHASE0_CHECKS`, `RUN_GOLD_CONTROL`, `RUN_AGENT`). It mirrors the
organizers' Getting-Started recipe (wheelhouse install, `VllmServer`, `swegemma.evaluate.Evaluator` with the
subprocess sandbox) and writes everything under `/kaggle/working/runs/<EXP_ID>/` in the layout that
`tools/analyze_traces.py` and `tools/estimate_budget.py` read.

**Attach as inputs** (Add Input): competition `gemma-4-developer-agent`, dataset
`metric/gemma-4-developer-agent-wheelhouse`, model `google/gemma-4/Other/gemma-4-31b-it-qat-w4a16-ct/2`
(or the `.../gemma-4-e4b-it-qat-w4a16-ct/2` / `.../gemma-4-e2b-it-qat-w4a16-ct/2` input when the zip's agent.yaml names a
small variant; the vLLM cell takes the model from the zip).
**Accelerator**: NVIDIA L4 x4 (competition-attached notebooks only). **Internet**: off.
The submission zip comes either from a private Kaggle dataset you upload (`SUBMISSION_ZIP`) or embedded as
base64 by `python tools/build_notebook.py --embed dist/submission.zip`.

**Status: UNTESTED on Kaggle.** This file was written without access to kaggle.com, the wheelhouse or the
harness source. Cells marked `# VERIFY` use API names taken from a community mirror of the starter notebook
and may need adjusting against the real `getting-started-gemma-4-developer-agent` notebook. Read the
starter notebook first and diff the server/evaluator cells.

Edit the parameters in the next cell, save a notebook *version* per experiment, then download
`/kaggle/working/runs/<EXP_ID>.zip` and run `python tools/analyze_traces.py runs/<EXP_ID>`.

In [ ]:
# ---------------------------------------------------------------- parameters (edit me)
EXP_ID = "EXP-032-p1"   # overridden by --set
RUN_PHASE0_CHECKS = True          # Q1/Q2/Q3/Q7/Q9 answers + HARNESS_README copy + sample_submission dump
RUN_GOLD_CONTROL = False   # overridden by --set
RUN_AGENT = True                  # run our submission on the sound tasks
MEASURE_DECODE_SPEED = True       # Q10: one timed completion against the server

# Task selection: explicit ids win; otherwise the first N_TASKS of the (optionally repo-filtered) dataset.
TASK_IDS = ["fastapi_14512", "fastapi_14609", "fastapi_14873", "fastapi_15280", "rich_3063", "rich_3676", "rich_4070", "rich_4075"]   # overridden by --set
N_TASKS = 8   # overridden by --set
REPO_FILTER = ""                  # e.g. "Textualize/rich"; "" = all repos
EXCLUDE_IDS = []                  # known-unsound tasks from a previous gold control (experiments/README.md Q5)

# Where the submission comes from (first that exists wins). Upload dist/submission.zip as a private dataset,
# or embed it: python tools/build_notebook.py --embed dist/submission.zip
SUBMISSION_ZIP = "/kaggle/input/datasets/YOUR_KAGGLE_USERNAME/gemma4-sub-v1/submission.zip"   # VERIFY path
SUBMISSION_B64 = "UEsDBBQAAAAIAAAAIVytu2QJiQEAALYCAAAKAAAAYWdlbnQueWFtbH1STY/TQAy951dY2ksrNSUtsIfsaSW4caiAAxJaVd6JtxkxGQeP01J+PZ5JUIEDl0wm9vvwc+7g/ZdD3bxuYN/s7+vdrm7etiDMCq7HeKIEL8IDcCT4EIbHE0WFlfYE5/0bCMzjGpThE32frOIxlI6vBxQMgZZbYHfEDeTjeT7c0wao88ryBKt3ddMYl4/f4PF+AypEcNitt9XdYm7/l7mEA0HqcaSHzIXB/yQxy8G8Rr7AiKJePUfINoWw8/EEK8edwbySKVDStAGXLUoCjB04ji/+NAlm4Lp8oh/kJiXj1N4Ysp3PmbBkgyK+6MFgvME4mUMCFvAxqUwu8zwAnUmut+B8sjKk6fmI+Z5ezdoLmVcjuMSZcVtFm9OmvdDcXJXn0QVMqf038Kqj5MSPWbWFjzbzMvzIKad8zbrjshS4eO2tnnPO8dQcw/UWpSVj2Fj2U/yZ38FeCyr/B/PirAnV8I782cwbxEsRFE3b6jZjWwHUS77HEbVv/wzgt+r2ikP4b+esOvf9AlBLAwQUAAAACAAAACFc/PRllWwAAACBAAAAFQAAAGNvbmZpZ3Mvc2FtcGxpbmcueWFtbD3MQQ6DIBBG4T2nmBMYbKwpcxlidQqECoT+k3j8GhfuXr7Fg+xN+gLtwjQO1qA235js4J5XZ6Z5Mvty+KpoCo+apfyYXqN7GMRUcirBr7V8UmBDdNNbtyBgmqybT09l/eomHrFqiDgP6CrmD1BLAwQUAAAACAAAACFcOxEmQj0BAAD5AQAAGQAAAGNvbmZpZ3Mvc2FtcGxpbmdfbG9jLnlhbWxlkEtrhDAUhff+igNDoYUqMXVkdKC77gvddCcZzWiYeBPyYDr99Y0KpY9Vkku43znfDi/vrzl7YuCM13lZ5mzfQpteaPUpnYchfSvwJmaJIGcrnQjRyUcEYzu7HRcIDy9mqxWNxU3M+ogwKbqkZ3eKwygDKtbUyJ9RMl5B0JDtMIuPzsRgY+iCuUjyOJQNXz5xVh3gDQScFEO+RECiElTiTMYF3I+SlijKrMOmuoM54yq0ftwK8bSgeVhIaYvVgtALIhPgIqV6KWdKsFXnv6pH6idBoxyOK94utEQQAbPxAeKUIqNmuBo3+AVqnfEyIaIH3yMpkOu4N0OydJVaQ5FXg9xabU2L7IfLFmXBstVnC1Y0+/V+aVFX2T9H7bom+9bbGzqrsc3w13i7qk5zRb2Og+zCZOI4hbQhuCizL1BLAwQUAAAACAAAACFcxNYQABcCAACgAwAAEAAAAGV2YWxfY29uZmlnLnlhbWxlU0tv2zAMvvdXfEAPS4G4cNK0DVxst2E7dEAPPewWMDZtC5ElQ5Trdr9+lLy+MMCABZIivwd1ju+/H4ryaottub0pNpuivK5gGnbR1GQRPSgMeNx9EfAT2UPtXWs6rIQGRuunAI1OLBdriEfsGXHOdwSNaVsO8M6+wLico04bIwbmy7NzDPR8iN5rU7JWsN2j+IbdFVZXsF5j5g8HwTOusRR8xeZ6jdFOkptpffqJcZ1lveFH9NTc5WDtJxd1uBGdrzU9BW5wnJqOI6gOXgTacwEkawWTdShv8XP3xkQvNegmCqS9EtuQw9yY6HNrxToYp3DmnmLOfcBNdZx0xAtk1BlrkGsS5EWOFkx1n1ONwERBOyU41vqZXM0Xb/pMwQn2ZZJmU5b/aTONyaMbDL5htStXq057DQTG/f2vRbs7nJjHRbdUBGsGE9EGP+DIqmDOHI1r0rmmcZ10rdWxpVLRcAjKOpARVdI40SXR7KjqWKujj6qSMqrJqfvBPDHmQOOYa7XukeT0I3iFm/u88zMDH5KIkQUS6UXJVp+lTOGOESal9m9u9j1t4JxMXGy9xAPVJ503m9ijKORkxmJJFR1Fxmqf3FrU6TH7IFHhCif3KbVedmB7m61azvu0Tuq0fgSpdXEdfAvdPOUjyiI9iomi8a46AxIbP8WDsL6TRipsd6WGPy96pXvzGvzAvsL+NZpcrJLdZ38BUEsDBBQAAAAIAAAAIVwcIL0JxQcAAN4QAAARAAAAcHJvbXB0cy9lZGl0b3IubWRtWE2P3DYSvc+vKDiH2EG3xsnuwUkOC3vWxhoLOIY9wSKnFluiuomWSIWkpqcT5L/vqypKrXF8mpZEFuvj1avH+S1MZKIl48k+jjZmMlMOPgxhSpRCl8/81fqD89bGij7ZFPoHS/loaUo2kktpsnR2+Sjv0mD63qaMvVNsLDVH4w+83OCVyS51ziZdmU22Le3t0Ty4EKubm3u8Pbq2tZ4O0bQwbsax5/UXGKODy9S6rqMcyFAXbTpStGNILod4QQQtxckncjlROHvKcCNV9D6Ttw8wluxsKVqTgnf+UNFrxD2M+aKWUxNglv6wMeAT4SnaZollvYQjyCadCKY9uU4N96ExvRxMZ5NwTksh8qfiQkSajXwv4a78d4mQo/r2HOIpjaaxdUW/Jks1gto1YRgQYK2JrvtUs+GaU3KADbw/wg09CIcaGk0pCJ8QkZG97cOZ2kA+YI97sPDgm2/oPZfv5s8xhn1vh11rUxPdmF3wf938eXQ+p3/9JQs/FTtdDAMMR8umuUgJgcxP2+D7y/we2QgndiYvjpRQ97ZDEtndDVnTHHmJkfTaaH2G8zFXhPy4uPg/GN5H5xj8gWOXqlU36hZq9ZzPaEILYLlsN1dE2keDEjZTFNPZPuYXPyHg3vidQWzFwBs1YB9dygCGgmdZuL8uvKPnDUMc8THimuA7d5ii4ZQt65uStDdTe7C5VFqTYnr+deHmaQGsATAKoSe2mTZikv04muhtSjA/eYazR1h4hy05ZNNX9BtKjcaxnLoecM70w6uVJbzrMsMyr45Oo2RA0tpNfQ9fgAnjG6sHD1wU3XMpi9HK6WfgzOYd9+sE3KVjOKdVZqMdjONmIjOwt+qb6VNQB6VT3FA4wKlrFf2PAWtRJLRF7wYA+YjG3chybgEpAuMKO4QNACTubTxr17b4PeA9Oa/2S65WDTnDPQrI0rTHMTt0RnOs6B18BoEYZrng+a+E1MoJbBP+92CoKG2OVdkhYxoRTgOv6BFMjzn0rYb3/UvGZmfP2KeF0PRs5InqtQ81fDtrF35f0V3wOSKfm7k7jL9cCwr0GUWG8i2X62TtiLZBPQDLHril0K3qki7DPjCmkjt4lA6shW6LEf6lHJFePP7n/v4jaWGldwoCo8VyTw+mn6zkPNrfJwcD7OZ44UMGZnb2F4HC3euT9O3M383RNifObEh2PlXd6y8a+Q8Vfc5o95lXFsK62XKGl/b2ZoAroH3Xa/RMYYnufv306e2He2nrOcdcwx2vq0EpzN+gpMIGRgmAq12jZjt1CUT77wUommU5pnMRRM1urG0mcyn0z5aY5nlnB/C1m6uBaA8ghELX/FZ3b7hfh4BavXqJmqHH6XmdOAE7fqoltNr6Vh9fLPWIlyujycGMPUy06os8iXeTPwFaAF2yJgL5AS1eXJknBm09Pfv88e3d+3fv73b3v/z37YdntN3St99V4+XbWmrKZ4mv4ILGda4pJZSOewLFALIg4X7Zw+jmdZoEHWMLfc6WJcVcVp1dLpeSgus4qNWeN3LGYgDjX6C/9CuGIxrKYYCtWX9N5ErVSttLscVpzSdiZHc4c8x4V9M8TRV2j9xB4i97V0yMRrRNDNPhiL3R9qJpxNZTUBlEGfoStQRsWA0c2GWAtg3aDv+o6C2fu1/1Pr3CgW8uqjBecXzLFzNLEzE7TIhjni97jcC2AmBlNXiz0biW0vbuZFG2bvINTzByw4AFiIIbdEsfo8VMXuO/etI5eqYU/FrU0uAb4KTpp1YRA1WXZUiCQ2ZNuJaLo/FlQgwjlIhlKteRrry4ynpkgsSZTLX4jY/4cc0+ExmvvkMiMvCJIp7VMUGoJP8KX6je3jUui25RlitbhtBOPYj+g0i3Rq2x8uAE1gyaHTcLerr+bieaTh7A21nQD0Lg1AkAb3nVeJFVGDx4YvnG4mHeyB6/BuVDf0k2VYfRIViZRvVtHiDzZqw9EYWzi+eIei8GFNBQcOELUflOPqhA8H+XZg1rElZg8zgR2csN1BnXC7Ozcha4/rOiX7xVmt+ISoEp46KMcV8mLgfWySAtXbVhnb4QIe/qQiN6SAz9NGdKe/dWMs3RIE20PcnbnRDH9neVwVh+BHa3DT2rquqZ5FJTwgfteUQXk9cyUFWKMG8eUDCX5VProPKxvS7sKrgU3yQHSWmGtXzCFUj6xuzDpG01X2iEofX6g8HAN5mLxLqeJVMSisKuqyUdZZq/HEYJVxK+LY2sLqBQBjnjIg1AMtspJIUhjh5irpRpnyTMMdrtwol7YPkE6sLCgxfRl1dsxQVc0WbRUM5jTIFyACLemqTDvqJpTCm1YaphKbb0m16ONI3uyeUondw44pHdeF3WYUhZvcMV5hIbqiuWshRwMkdAZKVyt5LLj4JM5p0cyeXZboGXQjNyQZrHwt+COMBcOeUr4rfMCADgx5eYsTi+TVcd/HWt9wWxakc8XbMpHaxqX1WdlwqraIGzjIilVDoCwBlBVDSPBW1LwT7XJwgBF+m9XJDc9RZSbriYOthx0RvtPPPKaAEN8uw3CnPPo8wHv9Vbs4ppHPW5KI0pJ4cj1owDhXM7pcitdwswWFxe+WcYc/0CsViMYQyHhu/58p+I4loBHAYYm+dFRVHDjX0MmKlM3SEWxxaGEdUmPFPd/B9QSwMEFAAAAAgAAAAhXAxWDBT/AgAA/wUAABAAAABwcm9tcHRzL2xvY19hLm1khVTdT9swEH/PX3ESr6WaJrEHVHVCUEaliUkFtPFETXJprDp25LsQKsT/vjs7tExs2lvs3Nfv43wfejARwUBEUx0H73ZQhgrTEeMUbocAgRuM4w2BC2ELhkEugUyroV0gyyHu/rhm2+JEb3J6ZyLTFO5Dn7/BUoo9/3GxgJvl7eIUauurdFf3vmQbPIQIpTNEMDSBEMo+RvQMj9iYJ6uVNNoS9dpzR1pziMFvpK+UKkO3A8sE+GxK3mczPvO0KI6OYKmZxUsXw6PD9qFCKqPttPNr8dJYz/T1NQWueodUHOfpE0lDiFvrN8AhOG27VnoeautwnUBCabwPDIQmlk0aZ3e4jb2Xqx1Lj41wbLaCWVKFGG4I6hjahKyyEctELEcUoD7dsqEttEhkNpgKp3gvnNNbSKJkKgPfCWsiQRuI4SQPWxrncmBg46awksH3MV8+gbNeCnUimUYKk25QagfLDayJRboHDVmn1mv0VT5O4RqfRpuInUQwl0HpGBcBEmz9xyh99Ie44cy/9xZUAdUUhjVnWb9TN6MzmSVxxZtDJhA69ClHimzUM7WNxFNNtzzmJcUM0K7tOLT7HBTMlSg7uktjxVoyXWu4bDD7s0bDfRQn5+H1RhscmKpDtiHZjU+hNNFzbjDOpNJ7xCrb7tJ644Q+GjAWK+zUTsquSaskq6HMKjNqMHSEp+/1EedVAqkGsa1M3bme9r8/n4xDyW9d40k2hE1TtqfF5fK77NlMbTYvfl4tVnr6uG3KxHwylpqZ+fHscV6cn93daHjwglX2CH0pgw3NuPPp1UgC+jBkQf6xnFLq6uz628daicVWTKcalI3x4u9UaGOfRjX+tvmyDgmf1L1brRbXt7nSx6VXVvRPxsVh7DHRh8JipccoOWJE1fRwSuLoOyLvk9Qy2XjWkxXICiFhrxVEsfh1di4DzIijqCdWSA6cKLcHgxw2PCH43wDzQsysFqps8sXhoeycKSV9iJYRsri934oAPnnIWcoqsLziydn5gdFaZIZp8RtQSwMEFAAAAAgAAAAhXIdn0wnqAgAAfwUAABAAAABwcm9tcHRzL2xvY19iLm1kXVRRTxsxDH6/X2GJl00qN9i0PTDUaRpFQ0ITgk4aT21653JRc8kpTjgqxH+f7bRl7KVqfI79+fP35T5kMBHBQETTHgfvttCEFvWIsYb5GCCkDuMuQuBC2IBJwEEg00vqEMimELdvwsn2OJFIuT6YmKiG+5DLf7CkufPZ3fzuDNbWt3rGJ0vJ+gdISInDDmEdon5ao0mZwcp/S5SRofoGoyfuw9cFCBceO25qE2kFAkOEHJWEsbNNBx26QQbhMlvIhHVVHR3BlRSsnocYVg77RYvURDskG/xL9dxZn+jbiybeZodUHZdJlLAxxI0iDsHJWEuhaiHQlzowNMb7kIDQRO4vSLav0Zg9h7aJezww32aDZerBpI5gHUOvA7c2YqMkp4g8v9doMrSBHonMA37lLb6SxjgyZeMY31KJ+CC/i/M+tDzAtB62S2BeDZDtrTOR8zyvrX1Lecned1PSa579N6Fstg/c7XOZu+FWpIkhGVfDLXNwyPlyAs56JGDiNZMX5kazJRht6mBJiRWxkJSl0rNE35ZjDb/wcac+Bjt2wRV+BIb2UGC9bJ707to+iUqYOhu59bvS+JRxMYz3E8kv0wSPwkBihReleN6P8MeZ8n2FnXm0suX/x78IoJvbtydOahhW9o0I5o1A+8EZ6xnaKuRUw3f/r5v4+2AZnU1FhJfWG8dT0IixusVBxCUEmb22ZUCpIHJDR8VgSvHpxxMRYsteULi8f15if1aJweDy6np2BueiqWm1s9x5HmTUT0U0sny+K6QIY4CGpUpmK53UUWqXYqZp9XN2fTO7lSJ6D8L6wPtkb7CJsHvYDLM77uSp2mJ8haIxZCfeZSdOq9mfm9mP+eyCCz8al6UapcgYSjXWScqkL5T4V1Dp5rh9mXj3QjQdNpvJq3lKJy5wiOi9SeG/5XwTTZN4IQLucJpWV2u1amuV9vJK/eOyCYzRpvKI7TjOfuPD6HVTzlJ5EEVjK7lUTC0lyYx19RdQSwMEFAAAAAgAAAAhXMws2LfbAgAAnQUAABAAAABwcm9tcHRzL2xvY19jLm1klVRNb9swDL37VxDoNTX2ge0QDB2Kdh2CFSmQtoeeYsWmYyGyaIhys6Dofx8pJWmDYYfdJErie3x81BONYAKCgYCmOSfvdlBTg2mLoYSHLQHFDsM+wuCINmAiSBDY9Hp1ILaRwu4kHG2PE43k54MJkUt4ojGvwTJcXd7e/ljcw+X8Gq7u5jezn4+Ly4fZ3XwKrfVNypVf99SMDlkikrA2zgEFsP1AISMmzrqwzKNufY3Bs+BLGuN30GBrRhcnesTReFlRIiJUowDkxORjIMdgI8MKO/Nshe4EmPJ5QuqMXyM0JGw8RViJLhs96cuiODuDmRIoXoZAK4f9skGugx2iJf9avHTWR/7+mi4utKDiPCuShN9S2Fi/hkjkVJ5KJV+21mGVhJPCvUIymlB3qbTdWzSMXiuNgrGWvpkNgj6VGmPH0AbqE//GBqxTs2JA0cunaDS8gR6ZzRqnKTKYeiMbqJZL621cLsthV4kUduWU474hWWDGGCXIKulBXz5cKaXGR0Z1Qk8c4UuuT5vICZ6icSUspNbjna8fQFBE4UFaozcFx23NjmFrYweVIIS41CtVIlChb/K2hDk+790qrt525LIOSuOaUsvSWUTB0QNOGZp8VNOwS+W3o6+1aSem6gdnrJcHKxpjCXcnc9FkkyjOvdLLVP8ppWqVZuVE0Fy+sxsUP0T6y+L2HaGJLj205BxtxUF4uE0h7faWh2fjRszmvLHeOKmXtxiKBQ5qOmVp0hDLU5VCdFAPATrOA5w4ffz0QQ3aSMsTCfFnS6GfFvshnsK3cVDKnwFliqx6A43YtFIDTo+C5oZ1wthGGDnNdC7uosh/gGT6/4nVJO8mNlWl3Z0kI2Wb6lTjb1PLszEEYZmluSgWs/tfgqqisYRRPo+pWEfyqwbQj1L/BnE4TqhpM4PDd3BRzNo0iy2NbwoKjWAjwlEhnxHQJzrOcu5slD92pXbMo6p52GzL4g9QSwMEFAAAAAgAAAAhXLWF2tXuAAAAiAEAABYAAABzdWJfYWdlbnRzL2VkaXRvci55YW1sVZDBbsMwCIbvfgp2rxNVq3bIfc8RUZskaDbODFbVPf3SZFu1E/p+EPw/gpkGoMhWqoukofJqXGSA901TsIVAS6uBwAooGut031VWbXSCpizzLthSiSCVgIm/qMKaUPQEYaHwoVAkbNMoEbRd8+/qFS0sncslUhpgppzRX/zr+erZ/Ceav13w/OaDORa12sLh7YUlpBYJuq5fa8mraX9k6HJ0P80xFDES0wGkCDkrJengADzUJlt3OybxYMI4Tpxop8emJ90qGz1xJhvV0JrueIQZ9xxuJqGK9nf5USee//s9NO0V85q233V3zMl9A1BLAwQUAAAACAAAACFcVrcovdYBAAAAAwAAFQAAAHN1Yl9hZ2VudHMvbG9jX2EueWFtbF2ST5PTMAzF7/kUYvZAO1On/1s2dw7cGE7cOqqjtp46srEUuuXTo6Swy3LIJJat59/TyxN8/v7VLdYrWC1WO7dcusW2gZIiwQSn4FNLIEGphkLYHk7BNhLHO0w4Qen54FPXIbfTBvRC9hQiiMljDL+oCNAL+V4JOOkl8HkGdhYy6kVMuyM4ldQNndUT3FK5SkZvKoNI4FHxgoVJBBTlCp194dlgAvvYt2S3sxKrmD5bi8CVsjbwt1znO8Rg/bDdL9x2v4Ec/PVB2hcGUSwKt6CXsXYsyP5iKGZTqQDmHAO1tiyiMHkorZ6f3Xq5n8F+88ntd+vpDCQB2pSOduD8Zv6juf85wnlk8w82CLvGIM0Cq22nG9tejPNCkhML2WRCqSvGjppB6IBVS+JLyBoSN/DNMnDj+F9vgVMqI/w/UX1RQ+ZWxvqpZz90j4sg0tspvMswrFtJBjwk4lMO5i3okBj6gbUUgwSlF62rzrRjA2eyrN3G7B9dUPcD1d02uNw5r1Vg0dL7B+eHP/lAXc+zJZxV5qOdumur/7NrxvCq1Gvu9XClewM5Ipt3TSlKUwG4t79vXJ1JD5ad9lLZKKmgvsoN71M4v2d41GQu2OUhpIPB1HfsYvUbUEsDBBQAAAAIAAAAIVw0PmfSQAEAAAMCAAAVAAAAc3ViX2FnZW50cy9sb2NfYi55YW1sXZExU8MwDIV3/wpxLDA4bdrSIcwMbBwTW051lMSHYwdLuTT8ehy3wMFkW5Kf3/t8C09vL3q738FuuzvqstTbhwpicAR3p3sQYmFA38DE2BEXEAmburWpH7xbHoGJwAVTY7Hg4KANEeZ+AeuNmxqqTfBCPmmw4MLgg6dCeRyoyrdOqiE20Y5ig6/gNYnrVXdtorOfFLOi9HSxUsCzQGt9w7lGZ8tifZebkF19j2PbkhFqoCWUKVIOEWkMMZmZexSwaXPNx0xR8sTcW9MnpfN6Jz+ypOjJ8xAachV0NAyoD3pfnrQV/YGi5wOWR21EWc8SJ3OJcnMlAEWxGWMYRuFNTlwMjfpPp8pgVJhknKR+p6WC0aFPeCQEx5UC0L/k86kjqRNTmVh15Cmi/Mita2u7vx4uNd4wDqNLyOpkJn+Z+gJQSwMEFAAAAAgAAAAhXDRg46RCAQAAEwIAABUAAABzdWJfYWdlbnRzL2xvY19jLnlhbWx9kT9PwzAQxXd/ikMsMDht2tIhzAxsiIktutqX1Kr/BPtMFT49jluBYGCx5Du/53e/u4Wntxe53m5gs97sZdvK9UMHMViCO3UPCq2lmAC9BhX8YMYckU3wDURC3Q+mPAzezo+QiMAG1WMzo7MwhAjn4wzGK5s19UXN5DlBYpwT+OCpER4ddVWlhKakopkW8w5ei7lcfJcmWvNJsTr+k+eZYTBeJ+BjiVSOCC7obGmpIENOVFs4DKSYFr2marRUNQ2YbYlXPknEbPx41S3BCw8wpXmgI36YkGMjijfZDkZyDuVObtuDNCzfkeV5h+1eKhbGJ45ZXSa6uYKApllNMbiJ06oO3jgt/kLqKh8RMk+Z+xPNHUwWfaHEIdjUCQD5s4B6G4n7gpZzEiN5KlS+7foLqd8ZLrW0SugmW6btS5i6OfEFUEsDBBQAAAAIAAAAIVxJnGmqywAAAHYBAAAYAAAAc3ViX2FnZW50cy9sb2NhbGl6ZS55YW1sjY8xbsMwDEV3n+KPLZCku7feoEN3g5Foi4AsGSSNwD19ZSdIxnYiQD4+8heauUeugbL8cEcTFx9CJrMeX6SUM+fPvdlFtqCyuNTS4zspM5QpnmvJ21OgBl0LyOGJYU0Ol5lPYArp4KVMIEQZR9amxULqqOPBKy/VxKtueAs1NoF423U2txPC/kw7QCUi1DLKtCrt77w/9DcVv+st1Wbdda14hbFZA2FOzpfO1utwJLW+A84P27CQpx6v4UdLNdBlozn/iV3/h4U79gtQSwECFAAUAAAACAAAACFcrbtkCYkBAAC2AgAACgAAAAAAAAAAAAAApAEAAAAAYWdlbnQueWFtbFBLAQIUABQAAAAIAAAAIVz89GWVbAAAAIEAAAAVAAAAAAAAAAAAAACkAbEBAABjb25maWdzL3NhbXBsaW5nLnlhbWxQSwECFAAUAAAACAAAACFcOxEmQj0BAAD5AQAAGQAAAAAAAAAAAAAApAFQAgAAY29uZmlncy9zYW1wbGluZ19sb2MueWFtbFBLAQIUABQAAAAIAAAAIVzE1hAAFwIAAKADAAAQAAAAAAAAAAAAAACkAcQDAABldmFsX2NvbmZpZy55YW1sUEsBAhQAFAAAAAgAAAAhXBwgvQnFBwAA3hAAABEAAAAAAAAAAAAAAKQBCQYAAHByb21wdHMvZWRpdG9yLm1kUEsBAhQAFAAAAAgAAAAhXAxWDBT/AgAA/wUAABAAAAAAAAAAAAAAAKQB/Q0AAHByb21wdHMvbG9jX2EubWRQSwECFAAUAAAACAAAACFch2fTCeoCAAB/BQAAEAAAAAAAAAAAAAAApAEqEQAAcHJvbXB0cy9sb2NfYi5tZFBLAQIUABQAAAAIAAAAIVzMLNi32wIAAJ0FAAAQAAAAAAAAAAAAAACkAUIUAABwcm9tcHRzL2xvY19jLm1kUEsBAhQAFAAAAAgAAAAhXLWF2tXuAAAAiAEAABYAAAAAAAAAAAAAAKQBSxcAAHN1Yl9hZ2VudHMvZWRpdG9yLnlhbWxQSwECFAAUAAAACAAAACFcVrcovdYBAAAAAwAAFQAAAAAAAAAAAAAApAFtGAAAc3ViX2FnZW50cy9sb2NfYS55YW1sUEsBAhQAFAAAAAgAAAAhXDQ+Z9JAAQAAAwIAABUAAAAAAAAAAAAAAKQBdhoAAHN1Yl9hZ2VudHMvbG9jX2IueWFtbFBLAQIUABQAAAAIAAAAIVw0YOOkQgEAABMCAAAVAAAAAAAAAAAAAACkAekbAABzdWJfYWdlbnRzL2xvY19jLnlhbWxQSwECFAAUAAAACAAAACFcSZxpqssAAAB2AQAAGAAAAAAAAAAAAAAApAFeHQAAc3ViX2FnZW50cy9sb2NhbGl6ZS55YW1sUEsFBgAAAAANAA0ATAMAAF8eAAAAAA=="               # filled in by tools/build_notebook.py --embed
# Model: taken from the embedded zip's agent.yaml. Set MODEL_NAME only as a cross-check (the run asserts the
# zip declares exactly this name), e.g. --set MODEL_NAME='"gemma-4-e2b-it-qat-w4a16-ct"'. The kernel-metadata
# model_sources must attach google/gemma-4/other/<name>/2 for the same name.
MODEL_NAME = ""

# Compaction settings: Q7 ANSWERED [V] from HARNESS_README.md section 7.2 (interval 15, overlap 2,
# token_threshold 14,336, event_retention_size 5). The 'interval 5' reading confused retention with interval.
COMPACTION_INTERVAL = 15   # [V] HARNESS_README §7.2: interval 15, overlap 2, threshold 14,336, retention 5
COMPACTION_TOKEN_THRESHOLD = 14336

In [ ]:
# ---------------------------------------------------------------- environment + wheelhouse (hosts' recipe)
# Adapted from the organizers' Getting-Started notebook as mirrored by community repos.  # VERIFY
import glob, importlib, json, os, shutil, subprocess, sys, time, zipfile, io, base64
from pathlib import Path

os.environ.update({
    'LITELLM_LOCAL_MODEL_COST_MAP': 'True', 'TRANSFORMERS_NO_TF': '1', 'VLLM_WORKER_MULTIPROC_METHOD': 'spawn',
    'VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS': '1', 'VLLM_ENGINE_READY_TIMEOUT_S': '1200',
    'VLLM_NO_USAGE_STATS': '1', 'OTEL_SDK_DISABLED': 'true', 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
})
WHEELHOUSE_DIR = next(Path(p) for p in ['/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse',
                                        '/kaggle/input/gemma-4-developer-agent-wheelhouse'] if Path(p).exists())
for pat in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth', '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for pth in glob.glob(pat):
        try:
            os.unlink(pth)
        except OSError:
            pass
tmp_whl = Path('/tmp/wheelhouse'); tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    name = w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name
    if not (tmp_whl / name).exists():
        os.symlink(w, tmp_whl / name)
# The wheelhouse ships cp312 binary wheels (orchestrator env = Python 3.12), but the grading sandbox and
# Kaggle's CPU image are Python 3.13 (docker/Dockerfile.sandbox is `python:3.13-slim`). A single bulk
# `pip install ... check=True` aborts the WHOLE install when one wheel is incompatible, which is what killed
# the first EXP-000 push (apache_tvm_ffi cp312 on py3.13). Filter by interpreter tag, then assert that the
# packages we actually need for this run arrived.
# Import names, not distribution names: the `google_adk` wheel installs the `google.adk`
# namespace package, so importing 'google_adk' always fails. [V] google_adk-1.36.1.dist-info
REQUIRED_PKGS = ('swegemma', 'adk_submission', 'adk_eval_core', 'google.adk')

def _wheel_is_compatible(path: Path) -> bool:
    """True when the wheel's tags intersect this interpreter's supported tags."""
    try:
        from packaging.tags import sys_tags
        from packaging.utils import parse_wheel_filename
    except Exception:  # noqa: BLE001 - fall back to accepting everything
        return True
    try:
        _, _, _, tags = parse_wheel_filename(path.name)
    except Exception:  # noqa: BLE001 - odd filename: let pip decide
        return True
    supported = set(sys_tags())
    return any(t in supported for t in tags)

all_wheels = sorted(tmp_whl.glob('*.whl'))
usable = [w for w in all_wheels if _wheel_is_compatible(w)]
skipped = [w.name for w in all_wheels if w not in usable]
T_INSTALL = time.time()
proc = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall',
                       *(str(w) for w in usable)], capture_output=True, text=True)
if proc.returncode != 0:
    print('bulk install failed; retrying wheel-by-wheel to isolate the bad ones')
    failed = []
    for w in usable:
        r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', str(w)],
                           capture_output=True, text=True)
        if r.returncode != 0:
            failed.append(w.name)
    skipped += failed
importlib.invalidate_caches()
print(f'wheelhouse: installed {len(all_wheels) - len(skipped)}/{len(all_wheels)} wheels in {time.time() - T_INSTALL:.0f}s '
      f'(python {sys.version_info.major}.{sys.version_info.minor})')
if skipped:
    print(f'  incompatible/failed ({len(skipped)}):', ', '.join(sorted(skipped)[:12]))
missing = []
for _pkg in REQUIRED_PKGS:
    try:
        importlib.import_module(_pkg)
    except Exception as _exc:  # noqa: BLE001
        missing.append(f'{_pkg} ({_exc.__class__.__name__}: {_exc})')
assert not missing, 'required packages unavailable: ' + '; '.join(missing)
print('  required packages present:', ', '.join(REQUIRED_PKGS))

DATA_DIR = next(Path(p) for p in ['/kaggle/input/competitions/gemma-4-developer-agent',
                                  '/kaggle/input/gemma-4-developer-agent'] if Path(p).exists())
RUN_DIR = Path('/kaggle/working/runs') / EXP_ID
for sub in ('traces', 'patches', 'logs', 'phase0'):
    (RUN_DIR / sub).mkdir(parents=True, exist_ok=True)
print('data:', DATA_DIR, ' run dir:', RUN_DIR)

In [ ]:
# ---------------------------------------------------------------- Phase 0 checks (plan §3.4)
# Answers Q1/Q2/Q3/Q7/Q9 by reading the installed harness source; saves HARNESS_README.md and
# sample_submission/ to the run dir so they can be downloaded and read offline.
if RUN_PHASE0_CHECKS:
    import importlib.metadata as md, re
    report = {'python': sys.version, 'packages': {}}
    for pkg in ('swegemma', 'adk-submission', 'adk-eval-core', 'google-adk', 'vllm', 'litellm'):
        try:
            report['packages'][pkg] = md.version(pkg)
        except md.PackageNotFoundError:
            report['packages'][pkg] = None
    for name in ('HARNESS_README.md',):
        src = DATA_DIR / name
        if src.exists():
            shutil.copy(src, RUN_DIR / 'phase0' / name)
    ss = DATA_DIR / 'sample_submission'
    if ss.exists():
        shutil.copytree(ss, RUN_DIR / 'phase0' / 'sample_submission', dirs_exist_ok=True)
        print('--- sample_submission files (Q2) ---')
        for p in sorted(ss.rglob('*')):
            if p.is_file() and p.suffix in ('.yaml', '.yml', '.md', '.txt', '.json'):
                print(f'===== {p.relative_to(ss)}'); print(p.read_text()[:4000])

    def grep_pkg(pkg: str, patterns: list[str]) -> dict:
        hits = {}
        try:
            root = Path(importlib.import_module(pkg).__file__).parent
        except Exception as exc:  # noqa: BLE001
            return {'error': str(exc)}
        for pat in patterns:
            rx = re.compile(pat)
            for f in root.rglob('*.py'):
                for i, line in enumerate(f.read_text(errors='ignore').splitlines(), 1):
                    if rx.search(line):
                        hits.setdefault(pat, []).append(f'{f.relative_to(root)}:{i}: {line.strip()[:160]}')
        return hits

    report['Q1_tool_registry'] = grep_pkg('swegemma', [r'register\(', r'def (run_command|read_file|edit_file|write_file|get_status|submit_patch|get_code_neighbors|search_similar_code|get_code_subgraph)\b'])
    report['Q1_skill_tools'] = grep_pkg('adk_submission', [r'run_skill_script|load_skill_resource|SkillToolset'])
    report['Q2_eval_config_keys'] = grep_pkg('swegemma', [r'max_time_minutes|max_tool_calls|max_turns|timeout_seconds', r'load_submission_eval_config'])
    report['Q2_agent_tool_schema'] = grep_pkg('adk_submission', [r'agent_tool|config_path|skip_summarization'])
    report['Q3_thinking_forwarding'] = grep_pkg('adk_submission', [r'thinking_budget|thinking_token_budget|enable_thinking|include_thoughts'])
    report['Q7_compaction'] = grep_pkg('swegemma', [r'compaction_interval|token_threshold|EventsCompactionConfig'])
    report['Q9_extensions_limits'] = grep_pkg('swegemma', [r'ALLOWED_SUBMISSION_EXTENSIONS|MAX_SUBMISSION_SIZE_BYTES|allowed_file_extensions|max_total_size_bytes'])
    report['Q2_grading_timeout'] = grep_pkg('swegemma', [r'timeout', ]) if False else grep_pkg('swegemma', [r'verification.*timeout|timeout.*verif|command_timeout_seconds'])
    (RUN_DIR / 'phase0' / 'phase0_report.json').write_text(json.dumps(report, indent=1))
    print(json.dumps({k: (v if k == 'packages' else {kk: len(vv) for kk, vv in v.items()} if isinstance(v, dict) else v)
                      for k, v in report.items()}, indent=1))
    print('full grep hits saved to', RUN_DIR / 'phase0' / 'phase0_report.json')

In [ ]:
# ---------------------------------------------------------------- tasks + candidates
from swegemma.models import load_tasks  # [V] swegemma/models/__init__.py exports load_tasks(path)->list[Task]
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
all_tasks = load_tasks(TASKS_PATH)
from collections import Counter
print('repo counts:', Counter(t.repo for t in all_tasks))
if TASK_IDS:
    want = set(TASK_IDS)
    candidates = [t for t in all_tasks if t.instance_id in want]
else:
    pool = [t for t in all_tasks if (not REPO_FILTER or t.repo == REPO_FILTER) and t.instance_id not in set(EXCLUDE_IDS)]
    candidates = sorted(pool, key=lambda t: t.instance_id)[:N_TASKS]
print(f'{len(all_tasks)} tasks; candidates ({len(candidates)}):', [t.instance_id for t in candidates])

In [ ]:
# ---------------------------------------------------------------- unpack our submission
AGENT_DIR = Path('/kaggle/working/agent')
shutil.rmtree(AGENT_DIR, ignore_errors=True); AGENT_DIR.mkdir(parents=True)
if SUBMISSION_B64:
    zipfile.ZipFile(io.BytesIO(base64.b64decode(SUBMISSION_B64))).extractall(AGENT_DIR)
elif Path(SUBMISSION_ZIP).exists():
    zipfile.ZipFile(SUBMISSION_ZIP).extractall(AGENT_DIR)
elif not RUN_AGENT and (DATA_DIR / 'sample_submission').is_dir():
    # A gold-control run never calls a model, so it needs no submission of ours. Fall back to the
    # official sample_submission: it is a real, schema-valid submission dir, so the model/adapter/limits
    # validation below still runs for real instead of being skipped.
    shutil.copytree(DATA_DIR / 'sample_submission', AGENT_DIR, dirs_exist_ok=True)
    print('no submission supplied; using official sample_submission (RUN_AGENT=False)')
else:
    raise SystemExit('no submission: set SUBMISSION_ZIP to your uploaded dataset file or embed with tools/build_notebook.py')
print('agent files:', sorted(str(p.relative_to(AGENT_DIR)) for p in AGENT_DIR.rglob('*') if p.is_file()))
import yaml
EVAL = (yaml.safe_load((AGENT_DIR / 'eval_config.yaml').read_text()) or {}).get('evaluation', {}) \
    if (AGENT_DIR / 'eval_config.yaml').exists() else {}
if not RUN_AGENT:
    # The official sample eval_config is deliberately tiny (1 min / 10 tool calls) to demo the schema.
    # Gold control runs no agent loop, so those caps only risk failing a sound task for budget reasons
    # and corrupting the Q5 answer. Give the grader room; the agent budget is irrelevant here.
    EVAL = {**EVAL, 'timeout_seconds': 300, 'max_time_minutes': 60.0, 'max_tool_calls': 100}
print('eval_config:', EVAL)

In [ ]:
# ---------------------------------------------------------------- vLLM server (hosts' recipe)  # VERIFY
# CPU-safe imports only: a gold-control run (RUN_AGENT=False) never loads vLLM/litellm/torch, and on
# Kaggle's Python 3.13 CPU image those cp312 wheels are not installed at all.
from adk_submission import ModelRegistry, discover_adapters, discover_declared_models
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS, EvalConfig, build_submission_limits

# D-004 2026-11-05: TARGET was hard-coded to 31B; it now comes from the zip, so a small-variant candidate
# runs without editing this file. Mirror of tools/submission_rules.py ALLOWED_MODEL_NAMES (the notebook runs
# on Kaggle and cannot import tools/). 12B is excluded there and here: it needs vLLM --hf-overrides
# (HARNESS_README §3.1 MODEL_VLLM_EXTRA_ARGS), which this cell does not pass.
ALLOWED_MODELS = ('gemma-4-31b-it-qat-w4a16-ct', 'gemma-4-e4b-it-qat-w4a16-ct', 'gemma-4-e2b-it-qat-w4a16-ct')
# [V] adk_submission.discovery.discover_declared_models(submission_dir) -> set[str]; walks the root
# agent, sub_agents and agent_tools. There is no `validate_single_declared_model` anywhere in the
# wheelhouse -- that was an invented name. Exactly one model may be declared, so assert the set size.
declared = discover_declared_models(str(AGENT_DIR))
assert len(declared) == 1, f'submission must declare exactly one model, got {sorted(declared)}'
TARGET = next(iter(declared)).strip()
for _prefix in ('openai/', 'google/', 'hosted_vllm/', 'custom/'):   # [V] HARNESS_README §3.2 normalize_model_name
    if TARGET.startswith(_prefix):
        TARGET = TARGET[len(_prefix):]
        break
assert TARGET in ALLOWED_MODELS, f'declared model {TARGET!r} not in {ALLOWED_MODELS}'
assert not MODEL_NAME or MODEL_NAME == TARGET, f'MODEL_NAME={MODEL_NAME!r} but the zip declares {TARGET!r}'
MODEL_DIR = f'/kaggle/input/models/google/gemma-4/other/{TARGET}/2'   # [V] path layout EXP-006 resolved 'exact'
print('model:', TARGET, '->', MODEL_DIR)
adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
# EXP-000 gold control needs no model: it applies each task's reference patch and runs test_patch.
# Gate the server so a gold-control-only run costs ZERO GPU quota (plan §5.7) and can use a CPU notebook.
NEED_SERVER = RUN_AGENT or MEASURE_DECODE_SPEED
server = None
SERVER_START_S = None
target_id = None
models = ModelRegistry()

if NEED_SERVER:
    import litellm, torch
    from adk_submission import VllmConfig, VllmServer
    from google.adk.models.lite_llm import LiteLlm
    try:
        from adk_submission.resolvers.generation import install_litellm_reasoning_patch
        install_litellm_reasoning_patch()
    except Exception as exc:  # noqa: BLE001
        print('reasoning patch not installed:', exc)
    litellm.drop_params = True
    gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
    assert gpu_count >= 1, 'no GPU: select the L4 x4 accelerator'
    assert Path(MODEL_DIR, 'config.json').exists(), (
        f'{MODEL_DIR} missing: kernel-metadata.json model_sources must list google/gemma-4/other/{TARGET}/2')
    tp = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)
    t0 = time.time()
    server = VllmServer(VllmConfig(
        model=MODEL_DIR, port=8000, host='127.0.0.1', tool_call_parser='gemma4', reasoning_parser='gemma4',
        default_chat_template_kwargs={'enable_thinking': True}, max_model_len=32768,
        dtype='bfloat16' if torch.cuda.is_bf16_supported() else 'auto', gpu_memory_utilization=0.90,
        enable_auto_tool_choice=True, enable_lora=True, max_loras=8, max_lora_rank=128,
        tensor_parallel_size=tp, startup_timeout=60 * 20), adapter_manifest=adapters)
    server.start()
    SERVER_START_S = round(time.time() - t0, 1)
    print(f'vLLM up in {SERVER_START_S}s (tp={tp}, gpus={gpu_count}) at {server.base_url}')
    target_id = getattr(server, 'effective_model_path', MODEL_DIR)
    for alias in {next(iter(declared)), TARGET}:
        models.register(alias, LiteLlm(model=f'openai/{target_id}', api_base=server.base_url, api_key='EMPTY'))
else:
    print('vLLM server SKIPPED (RUN_AGENT and MEASURE_DECODE_SPEED are both False) - no GPU required.')

In [ ]:
# ---------------------------------------------------------------- Q10: decode speed probe
DECODE = None
if MEASURE_DECODE_SPEED:
    import urllib.request
    body = json.dumps({'model': target_id, 'messages': [{'role': 'user', 'content': 'Count from 1 to 400 separated by spaces.'}],
                       'max_tokens': 400, 'temperature': 0.0, 'chat_template_kwargs': {'enable_thinking': False}}).encode()
    req = urllib.request.Request(server.base_url.rstrip('/') + '/chat/completions', data=body,
                                 headers={'Content-Type': 'application/json'}, method='POST')
    t0 = time.time()
    with urllib.request.urlopen(req, timeout=600) as resp:
        out = json.loads(resp.read())
    dt = time.time() - t0
    ct = out.get('usage', {}).get('completion_tokens', 0)
    DECODE = {'completion_tokens': ct, 'seconds': round(dt, 2), 'tok_s': round(ct / dt, 1) if dt else None}
    print('decode probe:', DECODE)

In [ ]:
# ---------------------------------------------------------------- evaluator helpers
import asyncio, concurrent.futures
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig  # [V] google/adk/apps/_configs.py:50
from swegemma.evaluate import Evaluator

def run_sync(fn):
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as ex:
            return ex.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())

def make_config(results_dir: Path) -> EvalConfig:
    limits, gen = build_submission_limits()
    return EvalConfig(
        tasks_path=TASKS_PATH, snapshots_dir=DATA_DIR / 'snapshots', results_dir=results_dir,
        submission_dir=AGENT_DIR, models=models, sandbox='subprocess',
        timeout_seconds=int(EVAL.get('timeout_seconds', 300)),
        max_time_minutes=float(EVAL.get('max_time_minutes', 60.0)),
        max_tool_calls=int(EVAL.get('max_tool_calls', 100)),
        max_turns=int(EVAL['max_turns']) if EVAL.get('max_turns') is not None else None,
        limits=limits, generation_constraints=gen, adapter_manifest=adapters,
        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(compaction_interval=COMPACTION_INTERVAL, overlap_size=2,
                                                        token_threshold=COMPACTION_TOKEN_THRESHOLD, event_retention_size=5),
        graph_dir=str(DATA_DIR / 'graphs'), embeddings_dir=str(DATA_DIR / 'embeddings'),
        wheels_dir=DATA_DIR / 'wheels', verbose=False)

def record(r, task, t_start: float, t_end: float) -> dict:
    patch = r.agent_patch or ''
    dur = float(r.duration_seconds or 0)
    wall = t_end - t_start
    return {'instance_id': task.instance_id, 'repo': task.repo, 'resolved': bool(r.resolved),
            'status': getattr(r, 'status', None), 'test_exit_code': r.test_exit_code,
            'duration_s': round(dur, 1), 'task_wall_s': round(wall, 1),
            'setup_s': round(max(0.0, wall - dur), 1),   # everything outside the agent loop (setup + grading)  VERIFY
            'tool_calls': getattr(r, 'tool_calls', None), 'llm_calls': getattr(r, 'total_llm_calls', None),
            'total_tokens': getattr(r, 'total_tokens', None), 'error': getattr(r, 'error', None) or getattr(r, 'error_message', None),
            'patch_chars': len(patch), 'patch': patch, 'trace': str(getattr(r, 'trace_path', '') or getattr(r, 'trace_json_path', '') or ''),
            'test_tail': (r.test_output or '')[-1500:], 'max_time_minutes': EVAL.get('max_time_minutes'),
            'max_tool_calls': EVAL.get('max_tool_calls')}

def append_row(path: Path, row: dict) -> None:
    with path.open('a', encoding='utf-8') as fh:
        fh.write(json.dumps(row) + '\n')

In [ ]:
# ---------------------------------------------------------------- EXP-000 gold control (Q5)
# `sound` must exist even when the gold control is skipped, or the agent cell below raises NameError.
sound = list(candidates)
if RUN_GOLD_CONTROL:
    from adk_eval_core.tracing import SessionTrace  # [V] tracing/__init__.py re-exports it; no-arg ctor

    class GoldEvaluator(Evaluator):
        """Phase 1 returns the reference patch; no model call, nothing is shown to any agent."""
        async def _run_agent_sandbox(self, task, *args, **kwargs):  # [V] evaluate.py:194 -> tuple[str, str|None, SessionTrace]
            return task.patch, None, SessionTrace()

    gold = GoldEvaluator(make_config(RUN_DIR / 'gold_control'))
    gold_rows = []
    for i, t in enumerate(candidates, 1):
        t0 = time.time()
        r = run_sync(lambda: gold.evaluate_task(task=t, task_index=i, total_tasks=len(candidates)))
        row = {'instance_id': t.instance_id, 'repo': t.repo, 'mode': 'gold', 'resolved': bool(r.resolved),
               'exit': r.test_exit_code, 'status': getattr(r, 'status', None), 'seconds': round(time.time() - t0, 1), 'error': (getattr(r, 'error', '') or '')[:300],
               'tail': (r.test_output or '')[-600:]}
        gold_rows.append(row)
        append_row(RUN_DIR / 'gold_control.jsonl', row)
        print('gold', t.instance_id, 'resolved=', row['resolved'], f"{row['seconds']}s")
    failures = [x['instance_id'] for x in gold_rows if not x['resolved']]
    (RUN_DIR / 'gold_failures.txt').write_text('# gold-control failures (exclude from cohorts)\n' + '\n'.join(failures) + '\n')
    sound = [t for t, x in zip(candidates, gold_rows) if x['resolved']]
    print(f'gold control: {len(sound)}/{len(candidates)} sound; failures: {failures}')

In [ ]:
# ---------------------------------------------------------------- baseline agent eval
summary = {'exp_id': EXP_ID, 'server_start_s': SERVER_START_S, 'decode_probe': DECODE, 'eval_config': EVAL,
           'compaction': {'interval': COMPACTION_INTERVAL, 'token_threshold': COMPACTION_TOKEN_THRESHOLD},
           'tasks': [], 'resolved': 0}
if RUN_AGENT:
    ev = Evaluator(make_config(RUN_DIR / 'official'))
    for i, t in enumerate(sound, 1):
        t0 = time.time()
        r = run_sync(lambda: ev.evaluate_task(task=t, task_index=i, total_tasks=len(sound)))
        row = record(r, t, t0, time.time())
        append_row(RUN_DIR / 'task_results.jsonl', row)
        (RUN_DIR / 'patches' / f'{t.instance_id}.patch').write_text(row['patch'])
        for cand in (Path(row['trace']) if row['trace'] else None, RUN_DIR / 'official' / 'traces' / f'trace_{t.instance_id}.json'):
            if cand and cand.exists():
                shutil.copy(cand, RUN_DIR / 'traces' / f'trace_{t.instance_id}.json'); break
        summary['tasks'].append(t.instance_id); summary['resolved'] += int(row['resolved'])
        print(f"[{i}/{len(sound)}] {t.instance_id} resolved={row['resolved']} calls={row['tool_calls']} "
              f"agent={row['duration_s']}s setup={row['setup_s']}s err={(row['error'] or '')[:100]}")
    n = len(summary['tasks'])
    print(f"RESOLVED {summary['resolved']}/{n}" if n else 'no tasks run')
(RUN_DIR / 'summary.json').write_text(json.dumps(summary, indent=1))

In [ ]:
# ---------------------------------------------------------------- bundle for download + stop server
# Official logs may contain repository snippets: keep the bundle private, never commit it (plan §1.3).
for extra in (RUN_DIR / 'official' / 'logs',):
    if extra.exists():
        shutil.copytree(extra, RUN_DIR / 'logs', dirs_exist_ok=True)
shutil.make_archive(str(Path('/kaggle/working/runs') / EXP_ID), 'zip', RUN_DIR)
print('bundle:', Path('/kaggle/working/runs') / f'{EXP_ID}.zip')
try:
    server.stop()
except Exception:  # noqa: BLE001
    pass